# Notebook 18 · Day 05.2 short tuning público

Versión pública que carga el resumen persistido del bloque Day 05.2 y deja la evidencia lista para portfolio.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

In [2]:
import json
import pandas as pd

DAY052_ROOT = ARTIFACTS_PUBLIC / "metrics" / "day05_2"
RUN_SUMMARY_PATHS = sorted(DAY052_ROOT.glob("*_run_summary.json"))
if not RUN_SUMMARY_PATHS:
    raise FileNotFoundError("No hay run_summary público Day 05.2.")

RUN_SUMMARY_PATH = RUN_SUMMARY_PATHS[-1]
run_summary = json.loads(RUN_SUMMARY_PATH.read_text(encoding="utf-8"))
RUN_ID = run_summary["run_id"]
canonical_candidates = pd.read_csv(DAY052_ROOT / f"{RUN_ID}_canonical_candidates.csv")
phase2_trials = pd.read_csv(DAY052_ROOT / f"{RUN_ID}_phase2_trials.csv")

print({"run_id": RUN_ID, "canonical_candidates": canonical_candidates.shape, "phase2_trials": phase2_trials.shape})
pd.DataFrame([run_summary])

{'run_id': '20260306T_day05_run03', 'canonical_candidates': (2, 38), 'phase2_trials': (8, 9)}


,run_id,day_id,cutoff_date,base_champion_variant,base_champion_metrics,secondary_reference_variants,phase2_trials_total,phase2_finalists_total,finalist_variants,best_tuned_variant,best_tuned_metrics,close_decision,current_model_champion_variant,serving_default_decision,can_open_day05_5,next_block
0,20260306T_day05_run03,Day 05.2,2028-02-21,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,"{'top1_hit': 0.772503, 'top2_hit': 0.882643, '...","[V2_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1, V2_TRAN...",8,2,[V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALAN...,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,"{'run_id': '20260306T_day05_run03', 'cutoff_da...",keep_day051_champion,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,keep_frozen_pending_explicit_rollout_decision,True,Day 05.5


In [3]:
trials_view = phase2_trials.sort_values(
    ["cv_top2_hit_mean", "cv_bal_acc_mean", "cv_f1_pos_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)
trials_view

,run_id,base_champion_variant,trial_idx,params_json,cv_top2_hit_mean,cv_bal_acc_mean,cv_f1_pos_mean,cv_coverage_mean,folds_json
0,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,6,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.856295,0.770363,0.554541,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.837317603228810..."
1,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,3,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.855643,0.839267,0.573674,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.835144365104004..."
2,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,8,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.855163,0.786101,0.558273,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.833902514746973..."
3,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,1,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.854350,0.800171,0.572287,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.828624650729587..."
4,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,5,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.853024,0.823647,0.569741,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.827382800372555..."
5,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,7,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.852614,0.792685,0.559114,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.846010555728034..."
6,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,4,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.848818,0.762210,0.555515,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.822104936355169..."
7,20260306T_day05_run03,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,2,"{""class_weight"": ""balanced"", ""colsample_bytree...",0.844376,0.822002,0.579558,1.0,"[{""fold_idx"": 1, ""top2_hit"": 0.819310773051847..."


In [4]:
finalists_view = canonical_candidates.loc[
    canonical_candidates["model_variant"].isin(run_summary["finalist_variants"])
].sort_values(
    ["top2_hit", "balanced_accuracy", "f1_pos"],
    ascending=[False, False, False],
).reset_index(drop=True)
finalists_view

,run_id,cutoff_date,dataset_alias,dataset_path,base_champion_variant,lr_equivalent_variant,model_family,model_variant,variant_stage,balance_tag,...,gate_pass,delta_top2_vs_day051_champion,delta_bal_acc_vs_day051_champion,delta_coverage_vs_day051_champion,promotable_vs_day051_champion,promotion_decision,model_path,metadata_path,metrics_json_path,params_json
0,20260306T_day05_run03,2028-02-21,V2_TRANSPORT_ONLY,./data/public/day041/dataset_modelo_v2_transpo...,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,phase2_short_tuning,CLASS_WEIGHT_BALANCED,...,False,-0.000379,-0.011522,0.0,False,keep_day051_champion,./models/candidates/day05_2_short_tuning/v2_tr...,./models/candidates/day05_2_short_tuning/v2_tr...,./artifacts/public/metrics/candidates/20260306...,"{""class_weight"": ""balanced"", ""colsample_bytree..."
1,20260306T_day05_run03,2028-02-21,V2_TRANSPORT_ONLY,./data/public/day041/dataset_modelo_v2_transpo...,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,phase2_short_tuning,CLASS_WEIGHT_BALANCED,...,False,-0.006076,-0.015919,0.0,False,keep_day051_champion,./models/candidates/day05_2_short_tuning/v2_tr...,./models/candidates/day05_2_short_tuning/v2_tr...,./artifacts/public/metrics/candidates/20260306...,"{""class_weight"": ""balanced"", ""colsample_bytree..."


In [5]:
secondary_refs = pd.DataFrame(
    {
        "secondary_reference_variant": run_summary["secondary_reference_variants"],
        "current_model_champion_variant": run_summary["current_model_champion_variant"],
        "close_decision": run_summary["close_decision"],
    }
)
secondary_refs

,secondary_reference_variant,current_model_champion_variant,close_decision
0,V2_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,keep_day051_champion
1,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_CLASS_WEIG...,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,keep_day051_champion
